# Laboratorio 10 — Análisis de electromiografía de superficie (sEMG)

**Análisis adaptado a un único registro OpenSignals (BITalino).**

Este cuaderno conserva la secuencia didáctica del laboratorio: adquisición, control de calidad, filtrado, segmentación, características temporales y frecuenciales, y discusión de resultados.

> **Alcance importante:** se adjuntó un solo registro EMG (04/09/2026, 15:20:15.812; 1000 Hz; canal A1). Los dos archivos TXT recibidos son copias del mismo registro. Por ello, este análisis **no compara un baseline independiente con una prueba**. Los segmentos de menor actividad se estiman dentro del propio registro y se usan como referencia intra-registro. Las etiquetas reposo/contracción son aproximadas, no anotaciones clínicas.

## 1. Fundamento fisiológico

La sEMG registra la suma de los potenciales de acción de múltiples unidades motoras detectados desde la superficie de la piel. La amplitud suele aumentar con el reclutamiento y la frecuencia de descarga de unidades motoras. El espectro aporta información complementaria sobre el contenido frecuencial de la señal.

**Rangos orientativos:** banda útil aproximada de 20–450 Hz; el contenido exacto depende del músculo, electrodos, colocación y sistema de adquisición.

## 2. Instrumentación y adquisición

Cadena general: músculo → electrodos → amplificador diferencial → conversión ADC → OpenSignals.

El archivo incluye metadatos de frecuencia de muestreo, resolución, sensor y canal. La señal se convierte de cuentas ADC a mV usando la ecuación del sensor BITalino EMG (G = 1009; VCC = 3.3 V).

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import skew, kurtosis, spearmanr
from IPython.display import display

VCC, G_EMG = 3.3, 1009
RUTA_REGISTRO = "opensignals_0C4314247FC7_2026-09-04_15-20-06 (1)(1).txt"

# En Google Colab, selecciona el TXT si no está en el directorio de trabajo.
if not os.path.exists(RUTA_REGISTRO):
    try:
        from google.colab import files
        print("Sube el archivo TXT del registro EMG.")
        uploaded = files.upload()
        if uploaded:
            RUTA_REGISTRO = next(iter(uploaded))
    except ImportError:
        pass

assert os.path.exists(RUTA_REGISTRO), "No se encontró el TXT. Colócalo junto al notebook o súbelo en Colab."

def leer_opensignals(ruta, canal="A1"):
    with open(ruta, "r", encoding="utf-8") as f:
        lineas = f.readlines()
    idx = next(i for i, linea in enumerate(lineas) if "EndOfHeader" in linea)
    meta = list(json.loads(lineas[1].lstrip("# ").strip()).values())[0]
    datos = np.loadtxt(ruta, comments="#")
    col = meta["column"].index(canal)
    bits = meta["resolution"][col]
    return datos[:, col].astype(float), meta, bits

adc, meta, N_BITS = leer_opensignals(RUTA_REGISTRO)
FS = int(meta["sampling rate"])
emg_mV = ((adc / (2**N_BITS)) - 0.5) * VCC / G_EMG * 1000
t = np.arange(len(adc)) / FS

print(f"Sensor: {meta['sensor']} | canal: A1")
print(f"Fecha/hora: {meta.get('date')} {meta.get('time')}")
print(f"Frecuencia de muestreo: {FS} Hz | resolución: {N_BITS} bits")
print(f"Muestras: {len(adc):,} | duración: {t[-1]:.2f} s")
print(f"ADC: mínimo={adc.min():.0f}, máximo={adc.max():.0f}, media={adc.mean():.2f}")
print(f"EMG convertido: mínimo={emg_mV.min():.3f} mV, máximo={emg_mV.max():.3f} mV")

### 2.1 Visualización de la señal cruda

El ADC de 10 bits representa valores entre 0 y 1023. La conversión a mV permite interpretar la amplitud. Los valores extremos pueden indicar saturación, es decir, recorte de la señal por los límites del conversor.

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(t, emg_mV, lw=0.45)
plt.xlabel("Tiempo (s)"); plt.ylabel("EMG (mV)")
plt.title("Registro EMG crudo — canal A1")
plt.grid(alpha=.25); plt.show()

rango_adc = (0, 2**N_BITS - 1)
sat = (adc <= rango_adc[0]) | (adc >= rango_adc[1])
sat_1022 = (adc <= 0) | (adc >= 2**N_BITS - 2)
print(f"Saturación estricta (ADC=0 o 1023): {sat.sum()} muestras ({100*sat.mean():.3f} %)")
print(f"Criterio ampliado (ADC=0, 1022 o 1023): {sat_1022.sum()} muestras ({100*sat_1022.mean():.3f} %)")

**Interpretación de calidad:** la saturación debe cuantificarse antes de extraer características. Si existen muestras recortadas, la amplitud real pudo superar el rango medible; el filtrado posterior no recupera esa información perdida. No se dispone de un registro baseline independiente para estimar el ruido instrumental aislado.

## 3. Preprocesamiento

Se aplica un pasa-banda Butterworth de 20–450 Hz y un notch de 60 Hz para reducir artefactos de baja frecuencia, componentes de alta frecuencia e interferencia de red. El filtrado de fase cero es apropiado para análisis offline.

In [ ]:
assert 0 < 20 < 450 < FS/2, "La banda de filtrado debe estar por debajo de Nyquist."
sos = signal.butter(4, [20, 450], btype="bandpass", fs=FS, output="sos")
emg_bp = signal.sosfiltfilt(sos, emg_mV)
b_notch, a_notch = signal.iirnotch(60, Q=30, fs=FS)
emg_f = signal.filtfilt(b_notch, a_notch, emg_bp)

plt.figure(figsize=(14,4))
plt.plot(t, emg_f, lw=.45)
plt.xlabel("Tiempo (s)"); plt.ylabel("EMG filtrada (mV)")
plt.title("Señal sEMG después del pasa-banda 20–450 Hz y notch 60 Hz")
plt.grid(alpha=.25); plt.show()

### 3.1 Densidad espectral de potencia (Welch)

Welch divide la señal en segmentos, calcula periodogramas y los promedia. Así se obtiene una estimación más estable de cómo se distribuye la potencia en frecuencia.

In [ ]:
f, pxx = signal.welch(emg_mV - np.mean(emg_mV), fs=FS, nperseg=min(2048, len(emg_mV)))
plt.figure(figsize=(12,4))
plt.semilogy(f, pxx)
plt.xlim(0, FS/2); plt.xlabel("Frecuencia (Hz)"); plt.ylabel("PSD (mV²/Hz)")
plt.title("PSD de la señal cruda — método de Welch")
plt.grid(alpha=.25); plt.show()

## 4. Segmentación: actividad y menor actividad dentro del mismo registro

Como no hay baseline separado, se estima una referencia de baja actividad usando la distribución de la envolvente RMS del propio registro. Este procedimiento permite una segmentación exploratoria, pero no equivale a comparar una medición de reposo independiente con una de contracción.

In [ ]:
def envolvente_rms(x, fs=FS, ventana_ms=200):
    n = max(1, int(fs*ventana_ms/1000))
    return np.sqrt(np.convolve(x**2, np.ones(n)/n, mode="same"))

env = envolvente_rms(emg_f)
ref_baja = env[env <= np.percentile(env, 20)]
K = 3
umbral = ref_baja.mean() + K*ref_baja.std()
mascara = env > umbral

def limpiar_mascara(activo, fs=FS, hueco_min_s=.20, dur_min_s=.15):
    d = np.diff(np.r_[0, activo.astype(int), 0])
    ini, fin = np.flatnonzero(d==1), np.flatnonzero(d==-1)
    segs=[]
    for s,e in zip(ini,fin):
        if segs and s-segs[-1][1] < hueco_min_s*fs:
            segs[-1]=(segs[-1][0],e)
        else:
            segs.append((s,e))
    segs=[(s,e) for s,e in segs if e-s >= dur_min_s*fs]
    m=np.zeros(len(activo),dtype=bool)
    for s,e in segs: m[s:e]=True
    return m,segs

mascara_activa, segmentos = limpiar_mascara(mascara)
duraciones=np.array([(e-s)/FS for s,e in segmentos])
print(f"Umbral intra-registro: {umbral*1000:.2f} µV")
print(f"Segmentos de actividad detectados: {len(segmentos)}")
if len(duraciones):
    print(f"Duración media: {duraciones.mean():.2f} ± {duraciones.std():.2f} s")

plt.figure(figsize=(14,4))
plt.plot(t, emg_f, lw=.4, label="EMG filtrada")
plt.plot(t, env, lw=1, label="Envolvente RMS")
plt.axhline(umbral, color="black", ls="--", label="Umbral intra-registro")
for s,e in segmentos: plt.axvspan(s/FS,e/FS,alpha=.15)
plt.xlabel("Tiempo (s)"); plt.ylabel("Amplitud (mV)")
plt.title("Segmentación exploratoria de actividad muscular")
plt.legend(); plt.grid(alpha=.2); plt.show()

**Precaución:** los tramos no activos según el umbral se denominan aquí *menor actividad* y no reposo confirmado. El umbral depende de la distribución de este registro y puede clasificar erróneamente actividad débil o artefactos.

## 5. Extracción de características

Se calculan características temporales (MAV, RMS, iEMG, varianza, longitud de onda, cruces por cero, cambios de pendiente, actividad de umbral, asimetría y curtosis) y frecuenciales (MNF, MDF y proporción de potencia por bandas).

In [ ]:
def crear_ventanas(x, fs=FS, ventana_ms=250, solapamiento=.5):
    n=int(round(fs*ventana_ms/1000))
    paso=max(1,int(round(n*(1-solapamiento))))
    ini=np.arange(0,len(x)-n+1,paso)
    return np.stack([x[i:i+n] for i in ini]),ini

def caracteristicas(x, fs=FS, eps=None):
    if eps is None: eps=0.01*np.std(x)
    dx=np.diff(x)
    zc=np.sum((x[:-1]*x[1:]<0)&(np.abs(dx)>=eps))
    ssc=np.sum((dx[:-1]*dx[1:]<0)&((np.abs(dx[:-1])>=eps)|(np.abs(dx[1:])>=eps)))
    f,p=signal.welch(x,fs=fs,nperseg=min(256,len(x)))
    banda=(f>=20)&(f<=450); fb,pb=f[banda],p[banda]
    pt=pb.sum()+1e-12
    acum=np.cumsum(pb)
    return {
        "MAV":np.mean(np.abs(x)), "RMS":np.sqrt(np.mean(x**2)),
        "iEMG":np.sum(np.abs(x))/fs, "VAR":np.var(x,ddof=1),
        "WL":np.sum(np.abs(dx)), "ZC":zc, "SSC":ssc,
        "WAMP":np.sum(np.abs(dx)>eps),
        "Asimetria":skew(x), "Curtosis":kurtosis(x,fisher=True,bias=False),
        "MNF":np.sum(fb*pb)/pt,
        "MDF":fb[min(np.searchsorted(acum,pt/2),len(fb)-1)],
        "P_20_80":pb[(fb>=20)&(fb<80)].sum()/pt,
        "P_80_150":pb[(fb>=80)&(fb<150)].sum()/pt,
        "P_150_450":pb[(fb>=150)&(fb<=450)].sum()/pt
    }

VENTANA_MS, SOLAP = 250, .5
ventanas, inicios = crear_ventanas(emg_f, FS, VENTANA_MS, SOLAP)
n=ventanas.shape[1]
fraccion_activa=np.array([mascara_activa[i:i+n].mean() for i in inicios])
etiqueta=np.where(fraccion_activa>=.8,"actividad",
          np.where(fraccion_activa<=.05,"menor_actividad","transicion"))
df=pd.DataFrame([caracteristicas(w) for w in ventanas])
df.insert(0,"t_inicio_s",inicios/FS)
df["etiqueta"]=etiqueta
df["fraccion_activa"]=fraccion_activa
df["saturada"]=[sat_1022[i:i+n].any() for i in inicios]
print(f"Ventanas: {len(df)} de {n} muestras ({VENTANA_MS} ms, {int(SOLAP*100)} % solapamiento)")
display(df["etiqueta"].value_counts().to_frame("Número de ventanas"))
display(df.head())

### 5.1 Comparación descriptiva entre clases intra-registro

Se comparan ventanas etiquetadas como actividad y menor actividad. Esta comparación es exploratoria: las clases se derivan de un umbral sobre la misma señal y no de anotaciones independientes.

In [ ]:
df_clases=df[df.etiqueta.isin(["menor_actividad","actividad"])].copy()
resumen=df_clases.groupby("etiqueta")[["RMS","MAV","WL","ZC","MNF","MDF","Curtosis"]].agg(["mean","std","count"])
display(resumen.round(4))

plt.figure(figsize=(7,4))
df_clases.boxplot(column="RMS",by="etiqueta")
plt.suptitle(""); plt.title("RMS por clase estimada"); plt.ylabel("RMS (mV)")
plt.show()

plt.figure(figsize=(14,4))
plt.plot(df["t_inicio_s"],df["RMS"]*1000, lw=1)
plt.xlabel("Tiempo (s)"); plt.ylabel("RMS (µV)")
plt.title("Evolución del RMS por ventana")
plt.grid(alpha=.25); plt.show()

### 5.2 Saturación por ventana

Las ventanas que contienen valores cercanos a los límites del ADC deben interpretarse con cautela, especialmente para características de amplitud y forma.

In [ ]:
display(pd.crosstab(df["etiqueta"],df["saturada"],rownames=["Clase estimada"],colnames=["Ventana con saturación"]))
print(f"Ventanas con saturación: {df.saturada.sum()} de {len(df)} ({100*df.saturada.mean():.2f} %)")

## 6. Tendencia de características a lo largo del registro

Se grafica RMS, MNF y MDF en función del tiempo. Estos cambios pueden relacionarse con variaciones de activación, colocación del electrodo, movimiento o fatiga; por sí solos no demuestran fatiga muscular.

In [ ]:
fig,ax=plt.subplots(3,1,figsize=(13,8),sharex=True)
for a,col in zip(ax,["RMS","MNF","MDF"]):
    a.plot(df.t_inicio_s,df[col],".-",ms=3)
    a.set_ylabel(col); a.grid(alpha=.2)
ax[-1].set_xlabel("Tiempo (s)")
plt.suptitle("Características por ventanas a lo largo del registro")
plt.tight_layout(); plt.show()

## 7. Limitaciones e interpretación

- Se analizó un único registro de 44.1 s, de un solo canal y un solo sujeto.
- Los dos TXT proporcionados eran copias del mismo archivo; no se contó con un baseline independiente.
- La separación entre actividad y menor actividad se estimó automáticamente desde la envolvente RMS; las etiquetas no son verdad de referencia.
- Las muestras saturadas implican pérdida irreversible de amplitud.
- No es válido afirmar generalización a otros sujetos ni reportar validación cruzada por sujetos con este único registro.
- No se puede concluir fatiga muscular únicamente por una tendencia de MNF/MDF/RMS en una grabación breve sin intensidad controlada y protocolo de contracciones estandarizado.

## 8. Conclusiones

Completa las conclusiones a partir de las tablas y gráficas generadas al ejecutar el cuaderno:

1. Describe la duración, frecuencia de muestreo y rango ADC del registro.
2. Indica el porcentaje de saturación y su posible efecto sobre el análisis.
3. Resume cómo se comportó la envolvente RMS y cuántos segmentos de actividad se detectaron.
4. Compara RMS, MNF y MDF entre las clases intra-registro, aclarando que son etiquetas estimadas.
5. Explica por qué no puede hacerse una comparación experimental baseline–contracción ni concluir generalización o fatiga con este único registro.

## 9. Exportación de características

Se guarda una fila por ventana con las características calculadas, el tiempo, la etiqueta estimada y el indicador de saturación.

In [ ]:
df.to_csv("caracteristicas_emg_registro_unico.csv",index=False)
print("Archivo generado: caracteristicas_emg_registro_unico.csv",df.shape)
try:
    from google.colab import files
    files.download("caracteristicas_emg_registro_unico.csv")
except ImportError:
    pass